In [1]:
library(ggplot2)
library(dplyr)
# Load the required libraries
library(ggpubr)
# load dplyr and tidyr library

library(tidyr)
library(parallel)
library(future)
library(furrr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
setwd("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/analysis/sRNA_deseq/genric_regions")

In [3]:
data <- read.csv(gzfile("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/IntronExonfeatureCount/pacBio/list2/all_feture_count_list2.tsv.gz"), header=FALSE,sep='\t')
data

V1,V2,V3,V4,V5,V6,V7,V8
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<chr>
exon:None_transcript:ENSMUST00200004126_144021_144931,129S1_SvImJ#1#chr1,144021,144931,+,911,6.37,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon:None_transcript:ENSMUST00200004126_147103_147234,129S1_SvImJ#1#chr1,147103,147234,+,132,0.00,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon:None_transcript:ENSMUST00200004126_147843_147930,129S1_SvImJ#1#chr1,147843,147930,+,88,0.00,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon:None_transcript:ENSMUST00200004126_148183_148402,129S1_SvImJ#1#chr1,148183,148402,+,220,0.50,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon:None_transcript:ENSMUST00200004126_150196_150622,129S1_SvImJ#1#chr1,150196,150622,+,427,2.33,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon:None_transcript:ENSMUST00200004126_150624_151067,129S1_SvImJ#1#chr1,150624,151067,+,444,1.75,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon:None_transcript:ENSMUST00200006521_144031_144556,129S1_SvImJ#1#chr1,144031,144556,-,526,4.62,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon:None_transcript:ENSMUST00200006521_150350_150623,129S1_SvImJ#1#chr1,150350,150623,-,274,0.83,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon:None_transcript:ENSMUST00200004330_191124_191277,129S1_SvImJ#1#chr1,191124,191277,-,154,9.08,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts


In [4]:
 # Split name column into 
data <- data %>% separate(V1, c('V9', 'V10','V11','V12'), sep =":")
data



Warning message:
“Expected 4 pieces. Missing pieces filled with `NA` in 196948602 rows [1, 2, 3,
4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...].”


V9,V10,V11,V12,V2,V3,V4,V5,V6,V7,V8
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<chr>
exon,None_transcript,ENSMUST00200004126_144021_144931,NA,129S1_SvImJ#1#chr1,144021,144931,+,911,6.37,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon,None_transcript,ENSMUST00200004126_147103_147234,NA,129S1_SvImJ#1#chr1,147103,147234,+,132,0.00,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon,None_transcript,ENSMUST00200004126_147843_147930,NA,129S1_SvImJ#1#chr1,147843,147930,+,88,0.00,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon,None_transcript,ENSMUST00200004126_148183_148402,NA,129S1_SvImJ#1#chr1,148183,148402,+,220,0.50,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon,None_transcript,ENSMUST00200004126_150196_150622,NA,129S1_SvImJ#1#chr1,150196,150622,+,427,2.33,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon,None_transcript,ENSMUST00200004126_150624_151067,NA,129S1_SvImJ#1#chr1,150624,151067,+,444,1.75,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon,None_transcript,ENSMUST00200006521_144031_144556,NA,129S1_SvImJ#1#chr1,144031,144556,-,526,4.62,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon,None_transcript,ENSMUST00200006521_150350_150623,NA,129S1_SvImJ#1#chr1,150350,150623,-,274,0.83,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
exon,None_transcript,ENSMUST00200004330_191124_191277,NA,129S1_SvImJ#1#chr1,191124,191277,-,154,9.08,129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts


In [5]:
# Group by V8, V9, V10 and sum the V7 values
new_data <- data %>%
  group_by(V8, V9) %>%
  summarise(summed_V7 = sum(V7, na.rm = TRUE)) %>%
  ungroup()
new_data

write.csv(new_data, "list2_count.csv", row.names=FALSE)

`summarise()` has grouped output by 'V8'. You can override using the `.groups`
argument.


V8,V9,summed_V7
<chr>,<chr>,<dbl>
129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts,CDS,1116561.5
129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts,exon,7183952.8
129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts,five_prime_UTR,229013.5
129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts,intron,2753341.3
129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts,three_prime_UTR,5085918.5
129S1_SvImJ/129S1_SvImJ-12.5dpp.2.featureCounts,CDS,1704816.8
129S1_SvImJ/129S1_SvImJ-12.5dpp.2.featureCounts,exon,11054526.7
129S1_SvImJ/129S1_SvImJ-12.5dpp.2.featureCounts,five_prime_UTR,357834.0
129S1_SvImJ/129S1_SvImJ-12.5dpp.2.featureCounts,intron,4260580.4
